In [1]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from math import ceil,floor
from scipy import ndimage
import skimage
import nibabel as nib
import ast
from radiomics import featureextractor
import SimpleITK as sitk
from segment_anything import sam_model_registry
import torch
import torch.nn.functional as F
from skimage import transform

from dicom_utils import *

In [2]:
FOLDER = "/media/bendico765/Crucial X9/Materiali Tesista"
DF_PATH = f"{FOLDER}/lesions_df.csv"

# Utility fuctions

## Read the 3d shape of a ROI

In [2]:
def read_3d_rois(roi, slices_per_roi: int) -> dict:
    number_of_rois = int(roi.pixel_array.shape[0] / slices_per_roi)
    
    rois = [
        roi.pixel_array[
            i*slices_per_roi : (i+1) * slices_per_roi,
            :,
            :
        ][::-1, ::-1, :] # for each roi flip it on the vertical axis
        for i in range(number_of_rois)
    ]
    
    z_indexes = [
        [
            i
            for i in range(roi.shape[0]) if roi[i, :, :].flatten().any()
        ]
        for roi in rois
    ]
    
    y_indexes = [
        [
            i
            for i in range(roi.shape[1]) if roi[:, i, :].flatten().any()
        ]
        for roi in rois
    ]
    
    x_indexes = [
        [
            i
            for i in range(roi.shape[2]) if roi[:, :, i].flatten().any()
        ]
        for roi in rois
    ]

    return [
        {
            "z_indexes": z_index,
            "y_indexes": y_index,
            "x_indexes": x_index,
            "roi": roi
        }
        for z_index, y_index, x_index, roi in zip(z_indexes, y_indexes, x_indexes, rois)
    ]

In [3]:
def get_roi_size(roi: dict) -> tuple:
    if pd.isnull(roi):
        return roi
    else:
        x_off = max(roi["x_indexes"]) - min(roi["x_indexes"])
        y_off = max(roi["y_indexes"]) - min(roi["y_indexes"])
        z_off = max(roi["z_indexes"]) - min(roi["z_indexes"])
        return (z_off+1, y_off+1, x_off+1)

# Associate DICOM and ROI files

In [4]:
df = pd.read_excel(
    "/media/bendico765/Crucial X9/Materiali Tesista/Advanced-MRI-Breast-Lesions-DA-Clinical-Sep2024.xlsx",
    header = 1,
    index_col = 0
    )
"""
df = df[[
    "tumor/benign1",
    "pos1",
    "tumor/benign2",
    "pos2",
    "tumor/benign3",
    "pos3",
    "tumor/benign4",
    "pos4",
    "tumor/benign5",
    "pos5",
    "tumor/benign6",
    "pos6",
]]
"""
df.drop(["Unnamed: 2", "Unnamed: 18", "Unnamed: 31", "Unnamed: 44"], axis=1,inplace=True) # dropping empty columns

In [5]:
df

,age at MRI,reason for referral ID#,additional reason for referral ID#,breast implants,BIRADS,tumor/benign1,pos1,pathology1,GRADE1,ER [SII] 1,...,KI67[%] 3,tumor/benign4,pos4,pathology4,tumor/benign5,pos5,pathology5,tumor/benign6,pos6,pathology6
Patient ID,,,,,,,,,,,,,,,,,,,,,
AMBL-001,47.4,2,5,0,4,1.0,L-60.85,9.0,-1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMBL-002,41.3,3,NaN,1,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMBL-003,53.3,6,NaN,0,6,1.0,R-2.76,1.0,3,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMBL-004,51.3,3,NaN,1,2,0.0,L-51.98,11.0,-1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMBL-005,75.3,3,5,0,4,1.0,R-28.46,3.0,2,2.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AMBL-628,64.0,1,NaN,0,6,1.0,L12.81,3.0,1,strong,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMBL-629,32.4,2,NaN,0,2,0.0,L39.93,17.0,-1,-1,...,-1,0.0,R1.93,17.0,0.0,R-48.07,17.0,0.0,R-32.07,17.0
AMBL-630,55.5,3,NaN,0,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Assign T2 and ROI to each patient

In [6]:
# path of directory Advanced MRI Breast Lesions
directory = "/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions"

dcms = []
rois = []

for patient_id in df.index:
    # load the slices and get the 3d shape of the patient
    dcm_path = [roots for roots, dirs, files in os.walk(f"{directory}/{patient_id}") if "Registered Ax T2 FSE" in roots][0]
    dcms.append(dcm_path)
    
    # get the path to the roi file associated with the patient
    roi_path = [roots for roots, dirs, files in os.walk(f"{directory}/{patient_id}") if "ROI" in roots]
    if len(roi_path) != 0:
        roi_path = roi_path[0]
        rois.append(f"{roi_path}/1-1.dcm")
    else:
        rois.append(np.nan)
        
df["Registered Ax T2 FSE path"] = dcms
df["Roi path"] = rois

df

,age at MRI,reason for referral ID#,additional reason for referral ID#,breast implants,BIRADS,tumor/benign1,pos1,pathology1,GRADE1,ER [SII] 1,...,pos4,pathology4,tumor/benign5,pos5,pathology5,tumor/benign6,pos6,pathology6,Registered Ax T2 FSE path,Roi path
Patient ID,,,,,,,,,,,,,,,,,,,,,
AMBL-001,47.4,2,5,0,4,1.0,L-60.85,9.0,-1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...
AMBL-002,41.3,3,NaN,1,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,NaN
AMBL-003,53.3,6,NaN,0,6,1.0,R-2.76,1.0,3,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...
AMBL-004,51.3,3,NaN,1,2,0.0,L-51.98,11.0,-1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...
AMBL-005,75.3,3,5,0,4,1.0,R-28.46,3.0,2,2.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AMBL-628,64.0,1,NaN,0,6,1.0,L12.81,3.0,1,strong,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...
AMBL-629,32.4,2,NaN,0,2,0.0,L39.93,17.0,-1,-1,...,R1.93,17.0,0.0,R-48.07,17.0,0.0,R-32.07,17.0,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...
AMBL-630,55.5,3,NaN,0,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,NaN


## Create an entry for each dataframe

In [7]:
df.columns

Index(['age at MRI', 'reason for referral ID#',
       'additional reason for referral ID#', 'breast implants', 'BIRADS',
       'tumor/benign1', 'pos1', 'pathology1', 'GRADE1', 'ER [SII] 1',
       'PR [SII] 1', 'HER2 [SII] 1', 'isTN1', 'ER [%] 1', 'PR [%] 1',
       'HER2 [%] 1', 'KI67[%] 1', 'tumor/benign2', 'pos2', 'pathology2',
       'GRADE2', 'ER [SII] 2', 'PR [SII] 2', 'HER2 [SII] 2', 'isTN2',
       'ER [%] 2', 'PR  [%] 2', 'HER  [%] 2', 'KI67[%] 2', 'tumor/benign3',
       'pos3', 'pathology3', 'GRADE3', 'ER [SII] 3', 'PR [SII] 3',
       'HER [SII] 3', 'isTN3', 'ER [%] 3', 'PR  [%] 3', 'HER  [%] 3',
       'KI67[%] 3', 'tumor/benign4', 'pos4', 'pathology4', 'tumor/benign5',
       'pos5', 'pathology5', 'tumor/benign6', 'pos6', 'pathology6',
       'Registered Ax T2 FSE path', 'Roi path'],
      dtype='object')

In [8]:
# for each entry, reformat it by creating a row for each lesion column
l = [
    [
        [
            index,
            1,
            row["tumor/benign1"],
            row["GRADE1"],
            row["ER [SII] 1"],
            row["PR [SII] 1"],
            row["HER2 [SII] 1"],
            row["isTN1"],
            row["ER [%] 1"],
            row["PR [%] 1"],
            row["HER2 [%] 1"],
            row["KI67[%] 1"],
            row["pos1"],
            row["Registered Ax T2 FSE path"],
            row["Roi path"]
        ],
        [
            index,
            2,
            row["tumor/benign2"],
            row["GRADE2"],
            row["ER [SII] 2"],
            row["PR [SII] 2"],
            row["HER2 [SII] 2"],
            row["isTN2"],
            row["ER [%] 2"],
            row["PR  [%] 2"],
            row["HER  [%] 2"],
            row["KI67[%] 2"],
            row["pos2"],
            row["Registered Ax T2 FSE path"],
            row["Roi path"]
        ],
        [
            index,
            3,
            row["tumor/benign3"],
            row["GRADE3"],
            row["ER [SII] 3"],
            row["PR [SII] 3"],
            row["HER [SII] 3"],
            row["isTN3"],
            row["ER [%] 3"],
            row["PR  [%] 3"],
            row["HER  [%] 3"],
            row["KI67[%] 3"],
            row["pos3"],
            row["Registered Ax T2 FSE path"],
            row["Roi path"]
        ],
        [
            index,
            4,
            row["tumor/benign4"],
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            row["pos4"],
            row["Registered Ax T2 FSE path"],
            row["Roi path"]
        ],
        [
            index,
            5,
            row["tumor/benign5"],
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            row["pos5"],
            row["Registered Ax T2 FSE path"],
            row["Roi path"]
        ],
        [
            index,
            6,
            row["tumor/benign6"],
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            row["pos6"],
            row["Registered Ax T2 FSE path"],
            row["Roi path"]
        ]
    ]
    for index, row in df.iterrows()
]

l = [ e2 for e1 in l[:] for e2 in e1 ]
lesions_df = pd.DataFrame(
    l,
    columns = [
        "Patient ID", 
        "lesion idx", 
        "tumor/benign",
        "GRADE",
        "ER [SII]",
        "PR [SII]",
        "HER2 [SII]",
        "isTN",
        "ER [%]",
        "PR [%]",
        "HER2 [%]",
        "KI67 [%]",
        "pos", 
        "Registered Ax T2 FSE path", 
        "Roi path"
    ]
)

# substitute -1 with nan values
lesions_df["tumor/benign"] = lesions_df["tumor/benign"].replace(-1, np.nan)
lesions_df["pos"] = lesions_df["pos"].replace(-1, np.nan)

# subtract -1 from the indexes
lesions_df["lesion idx"] = pd.to_numeric(lesions_df["lesion idx"]) -1 

# drop any row which doesn't have a tumor label, a slice location or a roi path
lesions_df.dropna(subset=["tumor/benign", "pos", "Roi path"], inplace = True) 

# instead of a generic pos, create a column for the slice and one for the position
lesions_df["Slice Location"] = lesions_df["pos"].apply(lambda x: x[1:] if str(x) != "nan" else np.nan)
lesions_df["Breast"] = lesions_df["pos"].apply(lambda x: x[:1] if str(x) != "nan" else np.nan)
lesions_df.drop(columns=["pos"], inplace=True)

# cast the slice location column from string to int
lesions_df["Slice Location"] = pd.to_numeric(lesions_df["Slice Location"])

lesions_df

,Patient ID,lesion idx,tumor/benign,GRADE,ER [SII],PR [SII],HER2 [SII],isTN,ER [%],PR [%],HER2 [%],KI67 [%],Registered Ax T2 FSE path,Roi path,Slice Location,Breast
0,AMBL-001,0,1.0,-1,-1,-1,-1,NaN,-1,-1,-1.0,-1,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-60.85,L
1,AMBL-001,1,1.0,-1,-1,-1,-1,NaN,-1.0,-1.0,-1.0,-1,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-56.85,R
12,AMBL-003,0,1.0,3,-1,-1,-1,NaN,-1,-1,-1.0,-1,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-2.76,R
18,AMBL-004,0,0.0,-1,-1,-1,-1,NaN,-1,-1,-1.0,-1,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-51.98,L
24,AMBL-005,0,1.0,2,2.9,1.4,0,NaN,100,70,20.0,intermediate,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-28.46,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1180,AMBL-629,4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-48.07,R
1181,AMBL-629,5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,-32.07,R
1188,AMBL-631,0,1.0,1 to 2,strong,neg,neg,NaN,-1,0,0.0,5,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,33.93,L
1189,AMBL-631,1,1.0,-1,strong,strong,neg,NaN,90.0,90.0,0.0,-1,/media/bendico765/Crucial X9/MRI Lesions/manif...,/media/bendico765/Crucial X9/MRI Lesions/manif...,25.93,L


In [10]:
inconsistent_patients = [
    "AMBL-572",
    "AMBL-574",
    "AMBL-579",
    "AMBL-590",
    "AMBL-595",
    "AMBL-626",
    "AMBL-629",
    "AMBL-631"
]

inconsistent_df = lesions_df[lesions_df["Patient ID"].isin(inconsistent_patients)].copy()

# remove the incosistences from the dataset
lesions_df = lesions_df[~ lesions_df["Patient ID"].isin(inconsistent_patients)].copy()

## To each lesion (among the consistent ones) associate the corrisponding pixel data

In [11]:
lesion_pixel_array = []
z_indexes = []
y_indexes = []
x_indexes = []
z_offset_array = []
y_offset_array = []
x_offset_array = []

for _, row in lesions_df.iterrows():  
    if pd.isnull(row["Roi path"]):
        lesion_pixel_array.append(np.nan)
        z_indexes.append(np.nan)
        y_indexes.append(np.nan)
        x_indexes.append(np.nan)
        z_offset_array.append(np.nan)
        y_offset_array.append(np.nan)
        x_offset_array.append(np.nan)
        continue
        
    if row["lesion idx"] == 0: # if it is the firt lesion of the patient, open up the dicom files of MRI and ROI
        patient_slices = read_dicomdir(row["Registered Ax T2 FSE path"])
        patient_3d_model = get_3d_shape(patient_slices)
        slices_per_file = patient_3d_model.shape[0]
        slice_thickness = patient_slices[0].SliceThickness
        
        roi = pydicom.dcmread(row["Roi path"])
        rois = read_3d_rois(roi, slices_per_file)

    candidate_rois = []
    
    # check for each lesion if it is localized on the left of right side
    mean = lambda l: sum(l)/len(l) if len(l) != 0 else 0    
    for roi in rois: 
        side = row["Breast"]
        if mean(roi["x_indexes"]) < 256: # closer to the left part of the image
            if side == "L":
                candidate_rois.append(roi)
        else:
            if side == "R":
                candidate_rois.append(roi)
    
    if( len(candidate_rois) == 0 ):
        print(f'[{row["Patient ID"]}] [{row["Breast"]}] [{row["Slice Location"]}] No candidate ROI')
        lesion_pixel_array.append(np.nan)
        z_indexes.append(np.nan)
        y_indexes.append(np.nan)
        x_indexes.append(np.nan)
        z_offset_array.append(np.nan)
        y_offset_array.append(np.nan)
        x_offset_array.append(np.nan)
        continue
    
    if( len(candidate_rois) > 1 ):
        picked_roi = False
        for roi in candidate_rois:
            min_z = min(roi["z_indexes"])
            max_z = max(roi["z_indexes"])
            
            max_slice = max(patient_slices[min_z].SliceLocation, patient_slices[max_z].SliceLocation)
            min_slice = min(patient_slices[min_z].SliceLocation, patient_slices[max_z].SliceLocation)
            
            if min_slice <= row["Slice Location"] <= max_slice or (abs(row["Slice Location"] - max_slice) <= slice_thickness or abs(row["Slice Location"] - min_slice) <= slice_thickness):
                lesion_pixel_array.append(roi["roi"])
                z_indexes.append(roi["z_indexes"])
                y_indexes.append(roi["y_indexes"])
                x_indexes.append(roi["x_indexes"])
                
                z_offset, y_offset, x_offset = get_roi_size(roi)
                z_offset_array.append(z_offset)
                y_offset_array.append(y_offset)
                x_offset_array.append(x_offset)
                
                picked_roi = True
                break
        
        if picked_roi == False:
            raise Exception(f'[{row["Patient ID"]}] [{row["Breast"]}] [{row["Slice Location"]}] Impossible to pick a ROI')
    else:
        lesion_pixel_array.append(candidate_rois[0]["roi"])
        z_indexes.append(candidate_rois[0]["z_indexes"])
        y_indexes.append(candidate_rois[0]["y_indexes"])
        x_indexes.append(candidate_rois[0]["x_indexes"])
        
        z_offset, y_offset, x_offset = get_roi_size(candidate_rois[0])
        z_offset_array.append(z_offset)
        y_offset_array.append(y_offset)
        x_offset_array.append(x_offset)

lesions_df["Pixel array"] = lesion_pixel_array
lesions_df["z_indexes"] = z_indexes
lesions_df["y_indexes"] = y_indexes
lesions_df["x_indexes"] = x_indexes
lesions_df["z_offset"] = z_offset_array
lesions_df["y_offset"] = y_offset_array
lesions_df["x_offset"] = x_offset_array

In [14]:
inconsistent_df["Pixel array"] = None
inconsistent_df["z_indexes"] = None
inconsistent_df["y_indexes"] = None
inconsistent_df["x_indexes"] = None
inconsistent_df["z_offset"] = 0
inconsistent_df["y_offset"] = 0
inconsistent_df["x_offset"] = 0

def assign_roi_to_entry(index: int, roi: dict):
    inconsistent_df.at[index, "Pixel array"] = roi["roi"]
    inconsistent_df.at[index, "z_indexes"] = roi["z_indexes"]
    inconsistent_df.at[index, "y_indexes"] = roi["y_indexes"]
    inconsistent_df.at[index, "x_indexes"] = roi["x_indexes"]
    inconsistent_df.at[index, "z_offset"],inconsistent_df.at[index, "y_offset"],inconsistent_df.at[index, "x_offset"]   = get_roi_size(roi)

# AMBL-572
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-572/11-25-2005-NA-MRI BREASTS - Delayed contrast-51529/600.000000-Registered Ax T2 FSE-43029")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-572/11-25-2005-NA-MRI BREASTS - Delayed contrast-51529/500.000000-ROI-99161/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(834, rois[3]) # R-11
assign_roi_to_entry(835, rois[4]) # L-21
assign_roi_to_entry(836, rois[0]) # R-51
assign_roi_to_entry(837, rois[1]) # L-11
assign_roi_to_entry(838, rois[2]) # L-27

# AMBL-574
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-574/12-01-2005-NA-MRI BREASTS - Delayed contrast-93585/600.000000-Registered Ax T2 FSE-56269")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-574/12-01-2005-NA-MRI BREASTS - Delayed contrast-93585/500.000000-ROI-73859/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(846, rois[3]) # L16.76
assign_roi_to_entry(847, rois[0]) # R-27.24
assign_roi_to_entry(848, rois[1]) # R-27.4
assign_roi_to_entry(849, rois[2]) # R-31.24

# AMBL-579
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-579/12-21-2005-NA-MRI BREASTS - Delayed contrast-84296/600.000000-Registered Ax T2 FSE-17158")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-579/12-21-2005-NA-MRI BREASTS - Delayed contrast-84296/500.000000-ROI-89400/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(876, rois[1]) # R-40.57
assign_roi_to_entry(877, rois[0]) # R-42.57

# AMBL-590
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-590/02-08-2006-NA-MRI BREASTS - Delayed contrast-70942/600.000000-Registered Ax T2 FSE-54767")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-590/02-08-2006-NA-MRI BREASTS - Delayed contrast-70942/500.000000-ROI-91209/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(942, rois[3]) # L37.31
assign_roi_to_entry(943, rois[4]) # L-13.27
assign_roi_to_entry(944, rois[0]) # R-22.09
assign_roi_to_entry(945, rois[1]) # R-24.29
assign_roi_to_entry(946, rois[2]) # L7.71

# AMBL-595
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-595/04-12-2006-NA-MRI BREASTS - Delayed contrast-99337/600.000000-Registered Ax T2 FSE-16358")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-595/04-12-2006-NA-MRI BREASTS - Delayed contrast-99337/500.000000-ROI-96750/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(972, rois[0]) # L22.86
assign_roi_to_entry(973, rois[1]) # L-45.34
# change rois 1 breast label from right to left
inconsistent_df.at[973, "Breast"] = "L"
    
# AMBL-626
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-626/12-22-2006-NA-MRI BREASTS - Delayed contrast-30307/600.000000-Registered Ax T2 FSE-43489")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-626/12-22-2006-NA-MRI BREASTS - Delayed contrast-30307/500.000000-ROI-54119/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(1158, rois[0]) # R4.40
assign_roi_to_entry(1159, rois[1]) # R12.4

# AMBL-629
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-629/01-30-2007-NA-MRI BREASTS - Delayed contrast-40670/600.000000-Registered Ax T2 FSE-05708")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-629/01-30-2007-NA-MRI BREASTS - Delayed contrast-40670/500.000000-ROI-75632/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(1176, rois[0]) # L39.93
assign_roi_to_entry(1177, rois[1]) # R-2.07
assign_roi_to_entry(1178, rois[2]) # R-20.07
assign_roi_to_entry(1179, rois[3]) # R1.93
assign_roi_to_entry(1180, rois[4]) # R-48.07
assign_roi_to_entry(1181, rois[5]) # R-32.07

# AMBL-631
slices = read_dicomdir("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-631/02-21-2007-NA-MRI BREASTS - Delayed contrast-91039/600.000000-Registered Ax T2 FSE-75697")
patient_3d_model = get_3d_shape(slices)
slices_per_file = patient_3d_model.shape[0]

roi = pydicom.dcmread("/media/bendico765/Crucial X9/MRI Lesions/manifest-1728494830954/Advanced-MRI-Breast-Lesions/AMBL-631/02-21-2007-NA-MRI BREASTS - Delayed contrast-91039/500.000000-ROI-68853/1-1.dcm")
rois = read_3d_rois(roi, slices_per_file)

assign_roi_to_entry(1188, rois[0]) # L33.93
assign_roi_to_entry(1189, rois[1]) # L25.93

In [15]:
# combine the dataframes together
lesions_df = pd.concat([lesions_df, inconsistent_df])

del inconsistent_df
del df
del lesion_pixel_array
del slices
del rois
del patient_3d_model

## Export the dataset

In [17]:
path = "/media/bendico765/Crucial X9/Materiali Tesista"
folder_name = "Roi masks"

# create the directory for the numpy files
try:
    os.mkdir(f"{path}/{folder_name}")
except FileExistsError:
    pass

# save the numpy 3d models inside the folder
numpy_filepaths_array = []
for patient_id in lesions_df["Patient ID"].unique():
    # create the patient rois folder
    try:
        os.mkdir(f"{path}/{folder_name}/{patient_id}")
    except FileExistsError:
        pass

    # for each patient lesion, save it
    for index, (_, row) in enumerate(lesions_df[lesions_df["Patient ID"] == patient_id].iterrows()):
        # save the roi
        pixel_array = row["Pixel array"]
        numpy_filepath = f"{path}/{folder_name}/{patient_id}/ROI {index}.npy"
        np.save(numpy_filepath, pixel_array)
        numpy_filepaths_array.append(numpy_filepath)

with open(f"{path}/{folder_name}/README.txt", "w") as file:
    file.write("Maschere ROI in formato npy")

lesions_df["Roi mask Filepath"] = numpy_filepaths_array

In [18]:
lesions_df.head(2)

,Patient ID,lesion idx,tumor/benign,GRADE,ER [SII],PR [SII],HER2 [SII],isTN,ER [%],PR [%],...,Slice Location,Breast,Pixel array,z_indexes,y_indexes,x_indexes,z_offset,y_offset,x_offset,Roi mask Filepath
0,AMBL-001,0,1.0,-1,-1,-1,-1,NaN,-1,-1,...,-60.85,L,"[[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[82, 83, 84, 85, 86, 87]","[172, 173, 174, 175, 176, 177, 178, 179, 180, ...","[142, 143, 144, 145, 146, 147, 148, 149, 150, ...",6,42,23,/media/bendico765/Crucial X9/Materiali Tesista...
1,AMBL-001,1,1.0,-1,-1,-1,-1,NaN,-1.0,-1.0,...,-56.85,R,"[[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[82, 83, 84, 85]","[188, 189, 190, 191, 192, 193, 194, 195, 196, ...","[360, 361, 362, 363, 364, 365, 366, 367, 368, ...",4,46,17,/media/bendico765/Crucial X9/Materiali Tesista...


In [20]:
lesions_df.to_csv("/media/bendico765/Crucial X9/Materiali Tesista/lesions_df.csv")

# Treating Disconnected components and holes

In [22]:
def cc3d(lung_mask0):
    minimum_cc_sum = np.sum(lung_mask0)*0.4
    minimum_tot_sum = np.sum(lung_mask0)*0.65
    minimum_cc2_sum = np.sum(lung_mask0)*0.3
    lung_mask = lung_mask0.copy()

    # Let us create a binary mask.
    binary_mask = np.round(lung_mask).astype('int')

    # Now, we perform region labelling. This way, every connected component
    # will have their own colour value.
    labelled_mask, num_labels = ndimage.label(binary_mask)

    # Let us now remove all the too small regions.
    for lab in range(num_labels+1):
        if np.sum(lung_mask[labelled_mask == lab]) < minimum_cc_sum:
            lung_mask[labelled_mask == lab] = 0

    if np.sum(lung_mask) < minimum_tot_sum:
        lung_mask = lung_mask0.copy()
        for lab in range(num_labels+1):
            if np.sum(lung_mask[labelled_mask == lab]) < minimum_cc2_sum:
                lung_mask[labelled_mask == lab] = 0

    return np.round(lung_mask)

fill_voids = lambda mask: ndimage.binary_fill_holes(mask).astype(int)

In [24]:
df = pd.read_csv("/media/bendico765/Crucial X9/Materiali Tesista/lesions_df.csv", index_col=0)
target_path = "/media/bendico765/Crucial X9/Materiali Tesista"
folder_name = "Cleaned Roi masks"

# create the directory for the numpy files
try:
    os.mkdir(f"{target_path}/{folder_name}")
except FileExistsError:
    pass

cleaned_roi_filepaths = []
for patient_id in df["Patient ID"].unique():
    # create the directory for the patient roi files
    try:
        os.mkdir(f"{target_path}/{folder_name}/{patient_id}")
    except FileExistsError:
        pass
        
    for index, (_, row) in enumerate(df[df["Patient ID"] == patient_id].iterrows()):
        filepath = f"{target_path}/{folder_name}/{patient_id}/ROI {index}.npy"
        
        # load the lesion volume
        mask_volume = np.load(row["Roi mask Filepath"])
        
        # for each volume, fill gasps and remove disconnected componenets
        mask_volume = fill_voids(cc3d(ndimage.binary_dilation(mask_volume)))
    
        np.save(filepath, mask_volume)
        cleaned_roi_filepaths.append(filepath)

with open(f"{target_path}/{folder_name}/README.txt", "w") as file:
    file.write("Maschere ROI a cui sono state rimosse le componenti disconnesse e i buchi, in formato npy")

df["Cleaned Roi mask Filepath"] = cleaned_roi_filepaths
df.to_csv("/media/bendico765/Crucial X9/Materiali Tesista/lesions_df.csv")

In [ ]:
df = pd.read_csv("/media/bendico765/Crucial X9/Materiali Tesista/lesions_df.csv", index_col=0)

# get pixel spacing and slice thickness for each image
df["Pixel Spacing"] = [ float(read_dicomdir(dicom_path)[0].PixelSpacing[0]) for dicom_path in df["Registered Ax T2 FSE path"] ]
df["Slice Thickness"] = [ float(read_dicomdir(dicom_path)[0].SliceThickness) for dicom_path in df["Registered Ax T2 FSE path"] ]

df.to_csv("/media/bendico765/Crucial X9/Materiali Tesista/lesions_df.csv")

# Radiomics

In [3]:
df = pd.read_csv(DF_PATH, index_col=0)

## Utils

In [8]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
@torch.no_grad()
def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if len(box_torch.shape) == 2:
        box_torch = box_torch[:, None, :] # (B, 1, 4)

    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None,
        boxes=box_torch,
        masks=None,
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed, # (B, 256, 64, 64)
        image_pe=medsam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
        sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
        dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
        multimask_output=False,
        )

    low_res_pred = torch.sigmoid(low_res_logits)  # (1, 1, 256, 256)

    low_res_pred = F.interpolate(
        low_res_pred,
        size=(H, W),
        mode="bilinear",
        align_corners=False,
    )  # (1, 1, gt.shape)
    low_res_pred = low_res_pred.squeeze().cpu().numpy()  # (256, 256)
    medsam_seg = (low_res_pred > 0.5).astype(np.uint8)
    return medsam_seg

def get_medsam_seg(medsam_model, img_slice, seg_box):
    size = 1024
    img_3c = np.repeat(img_slice[:,:,None],3, axis=-1) # ( H, W, 3 )
    H, W, _ = img_3c.shape

    # resize to size x size
    img = transform.resize(img_3c, (size, size), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
    img = (img - img.min()) / np.clip(
        img.max() - img.min(), a_min=1e-8, a_max=None
    )  # normalize to [0, 1], (H, W, 3)

    # convert the shape to (3, H, W)
    img_tensor = torch.tensor(img).float().permute(2, 0, 1).unsqueeze(0).to(device) # (1,3,size,size)

    # transfer seg_box to sizexsize scale
    box = seg_box / np.array([W, H, W, H]) * size
    with torch.no_grad():
        image_embedding = medsam_model.image_encoder(img_tensor) # (1, 256, 64, 64)
    
    return medsam_inference(medsam_model, image_embedding, box, H, W)

MedSAM_CKPT_PATH = "sam_vit_b_01ec64.pth"
device = "cpu"
medsam_model = sam_model_registry['vit_b'](checkpoint=MedSAM_CKPT_PATH)
medsam_model = medsam_model.to(device)
medsam_model.eval()

/home/bendico765/Desktop/INFN/MRI-Breast/MedSAM/segment_anything/build_sam.py:144: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f, map_location=torc

Sam(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
    )


In [9]:
shape_features = [
    'original_shape_Elongation',
    'original_shape_Flatness',
    'original_shape_LeastAxisLength',
    'original_shape_MajorAxisLength',
    'original_shape_Maximum2DDiameterColumn',
    'original_shape_Maximum2DDiameterRow',
    'original_shape_Maximum2DDiameterSlice',
    'original_shape_Maximum3DDiameter',
    'original_shape_MeshVolume',
    'original_shape_MinorAxisLength',
    'original_shape_Sphericity',
    'original_shape_SurfaceArea',
    'original_shape_SurfaceVolumeRatio',
    'original_shape_VoxelVolume'
]

first_order_features = [
    'original_firstorder_10Percentile',
    'original_firstorder_90Percentile',
    'original_firstorder_Entropy',
    'original_firstorder_InterquartileRange',
    'original_firstorder_Kurtosis',
    'original_firstorder_Maximum',
    'original_firstorder_MeanAbsoluteDeviation',
    'original_firstorder_Mean',
    'original_firstorder_Median',
    'original_firstorder_Minimum',
    'original_firstorder_Range',
    'original_firstorder_RobustMeanAbsoluteDeviation',
    'original_firstorder_RootMeanSquared',
    'original_firstorder_Skewness',
    'original_firstorder_TotalEnergy',
    'original_firstorder_Uniformity',
    'original_firstorder_Variance'
]

glcm_features = [
    'original_glcm_Autocorrelation',
    'original_glcm_ClusterProminence',
    'original_glcm_ClusterShade',
    'original_glcm_ClusterTendency',
    'original_glcm_Contrast',
    'original_glcm_Correlation',
    'original_glcm_DifferenceAverage',
    'original_glcm_DifferenceEntropy',
    'original_glcm_DifferenceVariance',
    'original_glcm_Id',
    'original_glcm_Idm',
    'original_glcm_Idmn',
    'original_glcm_Idn',
    'original_glcm_Imc1',
    'original_glcm_Imc2',
    'original_glcm_InverseVariance',
    'original_glcm_JointAverage',
    'original_glcm_JointEnergy',
    'original_glcm_JointEntropy',
    'original_glcm_MCC',
    'original_glcm_MaximumProbability',
    'original_glcm_SumAverage',
    'original_glcm_SumEntropy',
    'original_glcm_SumSquares'
]

gldm_features = [
    'original_gldm_DependenceEntropy',
    'original_gldm_DependenceNonUniformity',
    'original_gldm_DependenceNonUniformityNormalized',
    'original_gldm_DependenceVariance',
    'original_gldm_GrayLevelVariance',
    'original_gldm_HighGrayLevelEmphasis',
    'original_gldm_LargeDependenceEmphasis',
    'original_gldm_LargeDependenceHighGrayLevelEmphasis',
    'original_gldm_LargeDependenceLowGrayLevelEmphasis',
    'original_gldm_LowGrayLevelEmphasis',
    'original_gldm_SmallDependenceEmphasis',
    'original_gldm_SmallDependenceHighGrayLevelEmphasis',
    'original_gldm_SmallDependenceLowGrayLevelEmphasis'
]

glrlm_features = [
    'original_glrlm_GrayLevelNonUniformity',
    'original_glrlm_GrayLevelNonUniformityNormalized',
    'original_glrlm_GrayLevelVariance',
    'original_glrlm_HighGrayLevelRunEmphasis',
    'original_glrlm_LongRunEmphasis',
    'original_glrlm_LongRunHighGrayLevelEmphasis',
    'original_glrlm_LongRunLowGrayLevelEmphasis',
    'original_glrlm_LowGrayLevelRunEmphasis',
    'original_glrlm_RunEntropy',
    'original_glrlm_RunLengthNonUniformity',
    'original_glrlm_RunLengthNonUniformityNormalized',
    'original_glrlm_RunPercentage',
    'original_glrlm_RunVariance',
    'original_glrlm_ShortRunEmphasis',
    'original_glrlm_ShortRunHighGrayLevelEmphasis',
    'original_glrlm_ShortRunLowGrayLevelEmphasis'
]

glszm_features = [
    'original_glszm_GrayLevelNonUniformity',
    'original_glszm_GrayLevelNonUniformityNormalized',
    'original_glszm_GrayLevelVariance',
    'original_glszm_HighGrayLevelZoneEmphasis',
    'original_glszm_LargeAreaEmphasis',
    'original_glszm_LargeAreaHighGrayLevelEmphasis',
    'original_glszm_LargeAreaLowGrayLevelEmphasis',
    'original_glszm_LowGrayLevelZoneEmphasis',
    'original_glszm_SizeZoneNonUniformity',
    'original_glszm_SizeZoneNonUniformityNormalized',
    'original_glszm_SmallAreaEmphasis',
    'original_glszm_SmallAreaHighGrayLevelEmphasis',
    'original_glszm_SmallAreaLowGrayLevelEmphasis',
    'original_glszm_ZoneEntropy',
    'original_glszm_ZonePercentage',
    'original_glszm_ZoneVariance'
]

ngtdm_features = [
    'original_ngtdm_Busyness',
    'original_ngtdm_Coarseness',
    'original_ngtdm_Complexity',
    'original_ngtdm_Contrast',
    'original_ngtdm_Strength'
]

dynamic_features = [
    'max_derivative_phase_2', 
    'max_derivative_phase_3', 
    'max_derivative_phase_4',
    'max_derivative_phase_5', 
    'mean_derivative_phase_2',
    'mean_derivative_phase_3', 
    'mean_derivative_phase_4',
    'mean_derivative_phase_5', 
    'median_derivative_phase_2',
    'median_derivative_phase_3', 
    'median_derivative_phase_4',
    'median_derivative_phase_5', 
    'std_derivative_phase_2',
    'std_derivative_phase_3', 
    'std_derivative_phase_4',
    'std_derivative_phase_5'
]

## AX T2 FSE

### Original

In [7]:
features = shape_features + first_order_features + glcm_features + gldm_features + glrlm_features + glszm_features + ngtdm_features

def get_radiomics_dataset(df):
    """
    Given the lesions dataframe, iterate over each lesion
    """
    for _, row in df.iterrows(): # iterate over all lesions
        # get patient data
        volume_3d = get_3d_shape(read_dicomdir(row["Registered Ax T2 FSE path"]))
        pixel_spacing = row["Pixel Spacing"]
        slice_thickness = row["Slice Thickness"]
        roi = np.load(row["Roi mask Filepath"])

        extractor = featureextractor.RadiomicsFeatureExtractor()

        # create the SITK image of the MRI
        sitk_img = sitk.GetImageFromArray(volume_3d)
        sitk_img.SetSpacing( (pixel_spacing, pixel_spacing, slice_thickness) )

        # create the SITK image of the lesion mask
        sitk_mask = sitk.GetImageFromArray(roi)
        sitk_mask.SetSpacing( (pixel_spacing,pixel_spacing, slice_thickness) )

        # extract the radiomic features
        try:
            extracted_features = extractor.execute(sitk_img, sitk_mask)
        except ValueError as e:
            print(f"[{patient_id}] Skipping error: {e}")
            continue

        # return the features plus the original info
        l = [ 
            value.item() 
            for key, value in extracted_features.items() if key in features
        ]
        yield row.to_list() + l

iterator = get_radiomics_dataset(df)
radiomics_df = pd.DataFrame(
    iterator,
    columns = [ _ for _ in df.columns] + features
)
radiomics_df.to_csv(f"{FOLDER}/t2_original_masks.csv")

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

### Preprocessed

In [8]:
features = shape_features + first_order_features + glcm_features + gldm_features + glrlm_features + glszm_features + ngtdm_features

def get_radiomics_dataset(df):
    """
    Given the lesions dataframe, iterate over each lesion
    """
    for _, row in df.iterrows(): # iterate over all lesions
        # get patient data
        volume_3d = get_3d_shape(read_dicomdir(row["Registered Ax T2 FSE path"]))
        pixel_spacing = row["Pixel Spacing"]
        slice_thickness = row["Slice Thickness"]
        roi = np.load(row["Cleaned Roi mask Filepath"])

        extractor = featureextractor.RadiomicsFeatureExtractor()

        # create the SITK image of the MRI
        sitk_img = sitk.GetImageFromArray(volume_3d)
        sitk_img.SetSpacing( (pixel_spacing, pixel_spacing, slice_thickness) )

        # create the SITK image of the lesion mask
        sitk_mask = sitk.GetImageFromArray(roi)
        sitk_mask.SetSpacing( (pixel_spacing,pixel_spacing, slice_thickness) )

        # extract the radiomic features
        try:
            extracted_features = extractor.execute(sitk_img, sitk_mask)
        except ValueError as e:
            print(f"[{patient_id}] Skipping error: {e}")
            continue

        # return the features plus the label and the patient id
        l = [ 
            value.item() 
            for key, value in extracted_features.items() if key in features
        ]
        yield row.to_list() + l

iterator = get_radiomics_dataset(df)
radiomics_df = pd.DataFrame(
    iterator,
    columns = [ _ for _ in df.columns] + features
)
radiomics_df.to_csv(f"{FOLDER}/t2_preprocessed_masks.csv")

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

### MEDSAM

In [17]:
features = shape_features + first_order_features + glcm_features + gldm_features + glrlm_features + glszm_features + ngtdm_features
lesions_id = []
ground_truth_filepath = []
medsam_segmentation_filepath = []
dice_coeff = []

def get_medsam_not_robustscaled_dataset(df):
    """
    Given the lesions dataframe, iterate over each lesion
    """
    for lesion_index, row in df.iterrows(): # iterate over all lesions
        volume_3d = get_3d_shape(read_dicomdir(row["Registered Ax T2 FSE path"]))
        pixel_spacing = row["Pixel Spacing"]
        slice_thickness = row["Slice Thickness"]
        roi = np.load(row["Cleaned Roi mask Filepath"])

        z_min, z_max, y_min, y_max, x_min, x_max = bbox_3D(roi)
        # computing the segmentation box for medsam
        seg_box = np.array([[
            x_min, # x_0
            y_min, # y_0
            x_max, # x_1
            y_max  # y_1
        ]])
        # compute medsam segmentation for each slice
        mask = np.zeros(volume_3d.shape)
        for z in range(z_min, z_max+1):
            mask[z] = get_medsam_seg(medsam_model, volume_3d[z], seg_box)

        # compute dice coefficient
        medsman_seg_filepath = f"{FOLDER}/medsam_t2_segmentations/lesion_{lesion_index}.npy"
        lesions_id.append(lesion_index)
        ground_truth_filepath.append(row["Cleaned Roi mask Filepath"])
        medsam_segmentation_filepath.append(medsman_seg_filepath)
        dice_coeff.append(get_dice(mask, roi))

        # save medmsam segmentation 
        with open(medsman_seg_filepath, "wb") as f:
            np.save(f, mask)
        
        extractor = featureextractor.RadiomicsFeatureExtractor()

        # create the SITK image of the MRI
        sitk_img = sitk.GetImageFromArray(volume_3d)
        sitk_img.SetSpacing( (pixel_spacing, pixel_spacing, slice_thickness) )

        # create the SITK image of the lesion mask
        sitk_mask = sitk.GetImageFromArray(mask)
        sitk_mask.SetSpacing( (pixel_spacing,pixel_spacing, slice_thickness) )

        # extract the radiomic features
        try:
            extracted_features = extractor.execute(sitk_img, sitk_mask)
        except ValueError as e:
            print(f"[{patient_id}] Skipping error: {e}")
            continue

        # return the features plus the label and the patient id
        l = [ 
            value.item() 
            for key, value in extracted_features.items() if key in features
        ]
        yield row.to_list() + l

iterator = get_medsam_not_robustscaled_dataset(df)
radiomics_df = pd.DataFrame(
    iterator,
    columns = [ _ for _ in df.columns] + features
)
radiomics_df.to_csv(f"{FOLDER}/t2_medsam_masks.csv")

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

In [18]:
segmentations_df = pd.DataFrame({
    "lesion_id": lesions_id, 
    "ground_truth_filepath": ground_truth_filepath, 
    "medsam_segmentation_filepath": medsam_segmentation_filepath, 
    "dice_coefficient": dice_coeff
    })
segmentations_df.to_csv(f"{FOLDER}/medsam_t2_segmentations.csv")
segmentations_df["dice_coefficient"].describe()

count    164.000000
mean       0.521492
std        0.125663
min        0.129299
25%        0.439874
50%        0.541033
75%        0.614341
max        0.788998
Name: dice_coefficient, dtype: float64

# Dynamic

In [3]:
df = pd.read_csv(DF_PATH, index_col=0)

## Assign Multiphase to each patient and compute temporal resolutions

In [8]:
tmp_dict = {
    "Patient ID": [],
    "TemporalResolution 1": [],
    "TemporalResolution 2": [],
    "TemporalResolution 3": [],
    "TemporalResolution 4": [],
    "TemporalResolution 5": []
}

df["Registered AX Sen Vibrant MultiPhase path"] = [
    [roots for roots, dirs, files in os.walk("/".join(row["Registered Ax T2 FSE path"].split("/")[:-1])) if "Registered AX Sen Vibrant MultiPhase" in roots][0]
    for _, row in df.iterrows()
]

for _, row in df.drop_duplicates("Patient ID").iterrows():
    patient_id = row["Patient ID"]
    path = row["Registered AX Sen Vibrant MultiPhase path"]

    tr1, tr2, tr3, tr4, tr5 = get_temporal_resolutions(path)

    tmp_dict["Patient ID"].append(patient_id)
    tmp_dict["TemporalResolution 1"].append(tr1)
    tmp_dict["TemporalResolution 2"].append(tr2)
    tmp_dict["TemporalResolution 3"].append(tr3)
    tmp_dict["TemporalResolution 4"].append(tr4)
    tmp_dict["TemporalResolution 5"].append(tr5)

temporal_resolutions_df = pd.DataFrame(tmp_dict)
temporal_resolutions_df.to_csv(f"{FOLDER}/temporal_resolutions.csv")
df.to_csv(f"{DF_PATH}")

## Original

In [5]:
temporal_res_df = pd.read_csv(f"{FOLDER}/temporal_resolutions.csv", index_col=0)

def get_dynamic_information(df, temporal_res_df): 
    # merge the two datasets
    joined_df = df.merge(temporal_res_df, on="Patient ID")

    for _, row in joined_df.iterrows():
        result = row.to_list()
        #result.append(row["Patient ID"])
        #result.append(row["tumor/benign"])
        
        # get the temporal resolutions between one pulse and the next one
        tr2 = row["TemporalResolution 2"]
        tr3 = row["TemporalResolution 3"]
        tr4 = row["TemporalResolution 4"]
        tr5 = row["TemporalResolution 5"]

        # from the multiphase get the max, mean, median and std values for each phase
        max_values, mean_values, median_values, std_values = get_patient_intensity_values(row["Registered AX Sen Vibrant MultiPhase path"], row["Roi mask Filepath"])

        for values in [max_values, mean_values, median_values, std_values]:
            phase1_value = values[0]
            phase2_value = values[1]
            phase3_value = values[2]
            phase4_value = values[3]
            phase5_value = values[4]

            # compute the derivative as (t_i - t_i-1)/delta_t
            phase5_derivative = (phase5_value - phase4_value)/tr5
            phase4_derivative = (phase4_value - phase3_value)/tr4
            phase3_derivative = (phase3_value - phase2_value)/tr3
            phase2_derivative = (phase2_value - phase1_value)/tr2

            result.append(phase2_derivative)
            result.append(phase3_derivative)
            result.append(phase4_derivative)
            result.append(phase5_derivative)

        yield result

iterator = get_dynamic_information(df, temporal_res_df)
dynamic_df = pd.DataFrame(
    iterator,
    columns = [_ for _ in df.columns] + [
        "TemporalResolution 1",
        "TemporalResolution 2",
        "TemporalResolution 3",
        "TemporalResolution 4",
        "TemporalResolution 5"
        ] + [
        "max_derivative_phase_2",
        "max_derivative_phase_3",
        "max_derivative_phase_4",
        "max_derivative_phase_5",
        "mean_derivative_phase_2",
        "mean_derivative_phase_3",
        "mean_derivative_phase_4",
        "mean_derivative_phase_5",
        "median_derivative_phase_2",
        "median_derivative_phase_3",
        "median_derivative_phase_4",
        "median_derivative_phase_5",
        "std_derivative_phase_2",
        "std_derivative_phase_3",
        "std_derivative_phase_4",
        "std_derivative_phase_5"]
)
dynamic_df.to_csv(f"{FOLDER}/original_dynamic.csv")

## Preprocessed

In [6]:
temporal_res_df = pd.read_csv(f"{FOLDER}/temporal_resolutions.csv", index_col=0)

def get_dynamic_information(df, temporal_res_df): 
    # merge the two datasets
    joined_df = df.merge(temporal_res_df, on="Patient ID")

    for _, row in joined_df.iterrows():
        result = row.to_list()
        
        # get the temporal resolutions between one pulse and the next one
        tr2 = row["TemporalResolution 2"]
        tr3 = row["TemporalResolution 3"]
        tr4 = row["TemporalResolution 4"]
        tr5 = row["TemporalResolution 5"]

        # from the multiphase get the max, mean, median and std values for each phase
        max_values, mean_values, median_values, std_values = get_patient_intensity_values(row["Registered AX Sen Vibrant MultiPhase path"], row["Cleaned Roi mask Filepath"])

        for values in [max_values, mean_values, median_values, std_values]:
            phase1_value = values[0]
            phase2_value = values[1]
            phase3_value = values[2]
            phase4_value = values[3]
            phase5_value = values[4]

            # compute the derivative as (t_i - t_i-1)/delta_t
            phase5_derivative = (phase5_value - phase4_value)/tr5
            phase4_derivative = (phase4_value - phase3_value)/tr4
            phase3_derivative = (phase3_value - phase2_value)/tr3
            phase2_derivative = (phase2_value - phase1_value)/tr2

            result.append(phase2_derivative)
            result.append(phase3_derivative)
            result.append(phase4_derivative)
            result.append(phase5_derivative)

        yield result

iterator = get_dynamic_information(df, temporal_res_df)
dynamic_df = pd.DataFrame(
    iterator,
    columns = [_ for _ in df.columns] + [
        "TemporalResolution 1",
        "TemporalResolution 2",
        "TemporalResolution 3",
        "TemporalResolution 4",
        "TemporalResolution 5"
        ] + [
        "max_derivative_phase_2",
        "max_derivative_phase_3",
        "max_derivative_phase_4",
        "max_derivative_phase_5",
        "mean_derivative_phase_2",
        "mean_derivative_phase_3",
        "mean_derivative_phase_4",
        "mean_derivative_phase_5",
        "median_derivative_phase_2",
        "median_derivative_phase_3",
        "median_derivative_phase_4",
        "median_derivative_phase_5",
        "std_derivative_phase_2",
        "std_derivative_phase_3",
        "std_derivative_phase_4",
        "std_derivative_phase_5"]
)
dynamic_df.to_csv(f"{FOLDER}/preprocessed_dynamic.csv")

## Medsam

In [19]:
temporal_res_df = pd.read_csv(f"{FOLDER}/temporal_resolutions.csv", index_col=0)

def get_dynamic_information(df, temporal_res_df): 
    medsam_segmentations_df = pd.read_csv(f"{FOLDER}/medsam_t2_segmentations.csv", index_col=0)[["lesion_id", "medsam_segmentation_filepath"]].set_index("lesion_id")
    
    # merge the two datasets
    joined_df = df.join(medsam_segmentations_df).merge(temporal_res_df, on="Patient ID")

    for _, row in joined_df.iterrows():
        medsam_segmentation_filepath = row["medsam_segmentation_filepath"]
        result = row.to_list()
        result.remove(medsam_segmentation_filepath)
        
        # get the temporal resolutions between one pulse and the next one
        tr2 = row["TemporalResolution 2"]
        tr3 = row["TemporalResolution 3"]
        tr4 = row["TemporalResolution 4"]
        tr5 = row["TemporalResolution 5"]

        # from the multiphase get the max, mean, median and std values for each phase
        max_values, mean_values, median_values, std_values = get_patient_intensity_values(row["Registered AX Sen Vibrant MultiPhase path"], medsam_segmentation_filepath)

        for values in [max_values, mean_values, median_values, std_values]:
            phase1_value = values[0]
            phase2_value = values[1]
            phase3_value = values[2]
            phase4_value = values[3]
            phase5_value = values[4]

            # compute the derivative as (t_i - t_i-1)/delta_t
            phase5_derivative = (phase5_value - phase4_value)/tr5
            phase4_derivative = (phase4_value - phase3_value)/tr4
            phase3_derivative = (phase3_value - phase2_value)/tr3
            phase2_derivative = (phase2_value - phase1_value)/tr2

            result.append(phase2_derivative)
            result.append(phase3_derivative)
            result.append(phase4_derivative)
            result.append(phase5_derivative)

        yield result

iterator = get_dynamic_information(df, temporal_res_df)
dynamic_df = pd.DataFrame(
    iterator,
    columns = [_ for _ in df.columns] + [
        "TemporalResolution 1",
        "TemporalResolution 2",
        "TemporalResolution 3",
        "TemporalResolution 4",
        "TemporalResolution 5"
        ] + [
        "max_derivative_phase_2",
        "max_derivative_phase_3",
        "max_derivative_phase_4",
        "max_derivative_phase_5",
        "mean_derivative_phase_2",
        "mean_derivative_phase_3",
        "mean_derivative_phase_4",
        "mean_derivative_phase_5",
        "median_derivative_phase_2",
        "median_derivative_phase_3",
        "median_derivative_phase_4",
        "median_derivative_phase_5",
        "std_derivative_phase_2",
        "std_derivative_phase_3",
        "std_derivative_phase_4",
        "std_derivative_phase_5"]
)
dynamic_df.to_csv(f"{FOLDER}/medsam_dynamic.csv")